In [ ]:
import kagglehub
fake_news_detection_dataset = kagglehub.dataset_download("emineyetm/fake-news-detection-datasets")
print(fake_news_detection_dataset)

Using Colab cache for faster access to the 'fake-news-detection-datasets' dataset.
/kaggle/input/fake-news-detection-datasets


In [ ]:
import os
import pandas as pd

dataset_path = fake_news_detection_dataset
news_dataset_subdir = os.path.join(dataset_path, 'News _dataset')

print(f"Files in the sub-directory ({news_dataset_subdir}):")
files_in_subdir = os.listdir(news_dataset_subdir)
for file_name in files_in_subdir:
    print(file_name)

csv_files = [f for f in files_in_subdir if f.endswith('.csv')]

if csv_files:
    data_true_file = os.path.join(news_dataset_subdir, csv_files[0])
    print(f"\nLoading data from (true.csv): {data_true_file}")
    data_fake_file = os.path.join(news_dataset_subdir, csv_files[1])
    print(f"\nLoading data from (fake.csv): {data_fake_file}")

    try:
        df_fake = pd.read_csv(data_fake_file)
        print("Dataset loaded successfully into a pandas DataFrame. (fake.csv)")
        print("First 5 rows of the dataset:")
        print(df_fake.head())
        print("Dataset information:")
        df_fake.info()

        print('''========================================
                 ========================================''')
        df_true = pd.read_csv(data_true_file)

        print("Dataset loaded successfully into a pandas DataFrame. (fake.csv)")
        print("First 5 rows of the dataset:")
        print(df_true.head())
        print("Dataset information:")
        df_true.info()
    except Exception as e:
        print(f"Error loading {data_fake_file}: {e}")
else:
    print(f"No CSV files found in the '{news_dataset_subdir}' directory. Please specify which file to load.")

Files in the sub-directory (/kaggle/input/fake-news-detection-datasets/News _dataset):
True.csv
Fake.csv

Loading data from (true.csv): /kaggle/input/fake-news-detection-datasets/News _dataset/True.csv

Loading data from (fake.csv): /kaggle/input/fake-news-detection-datasets/News _dataset/Fake.csv
Dataset loaded successfully into a pandas DataFrame. (fake.csv)
First 5 rows of the dataset:
                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   
3  On Christmas

In [ ]:
df_fake['class']=0
df_true['class']=1
concatenated_dataset = pd.concat([df_true, df_fake])
concatenated_dataset.head()

,title,text,subject,date,class
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",1
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",1
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",1
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",1
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",1


In [ ]:
#droping unnecessary columns
concatenated_dataset = concatenated_dataset.drop(['title', 'subject', 'date'], axis=1)
concatenated_dataset.head()

,text,class
0,WASHINGTON (Reuters) - The head of a conservat...,1
1,WASHINGTON (Reuters) - Transgender people will...,1
2,WASHINGTON (Reuters) - The special counsel inv...,1
3,WASHINGTON (Reuters) - Trump campaign adviser ...,1
4,SEATTLE/WASHINGTON (Reuters) - President Donal...,1


In [ ]:
#reset index, after remove it from the dataset (column)
concatenated_dataset = concatenated_dataset.reset_index(drop=True)
concatenated_dataset.head()

,text,class
0,WASHINGTON (Reuters) - The head of a conservat...,1
1,WASHINGTON (Reuters) - Transgender people will...,1
2,WASHINGTON (Reuters) - The special counsel inv...,1
3,WASHINGTON (Reuters) - Trump campaign adviser ...,1
4,SEATTLE/WASHINGTON (Reuters) - President Donal...,1


I saw a video on YouTube about how to clean text for this specific dataset, so I applied what I learned from it.

In [ ]:
import re
import string

def clean_text(text):
    text = text.lower()
    # Supprime les caractères spéciaux comme les crochets `[]` et les guillemets `"?`.
    for char in ['[',']','?']:
        text = text.replace(char,'')
    # Supprime les URL commençant par 'https://' ou 'http://'.
    text = re.sub(r"https?://\S+/www\.\S+", "", text)
    # Supprime les balises HTML (ex: '<tag>')
    text = re.sub(r"<[^>]+>", "", text)
    # Supprime les signes de ponctuations
    for char in string.punctuation:
        text = text.replace(char,'')
    # Supprime les caractères de nouvelle ligne.
    text = re.sub(r"\n", "", text)
    # Supprimer des séquences de type 'mot-chiffre-mot
    text = re.sub(r"\w\d\w", "", text)
    return text

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
#Apply text cleaning
concatenated_dataset['text'] = concatenated_dataset['text'].apply(clean_text)
#train test split
X = concatenated_dataset['text']
y = concatenated_dataset['class']
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)
# Initialize the CountVectorizer for Bag of Words
vectorizer = CountVectorizer()

x_train = vectorizer.fit_transform(X_train)
x_test = vectorizer.transform(X_test)


In [ ]:
from sklearn.naive_bayes import MultinomialNB

multNB = MultinomialNB()
multNB.fit(x_train, y_train)
y_pred = multNB.predict(x_test)

print('Train Precision: ',multNB.score(x_train, y_train))
print('Test Precision: ',multNB.score(x_test, y_test))

Train Precision:  0.9639734951834734
Test Precision:  0.9547884187082405


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.95      0.96      4650
           1       0.94      0.96      0.95      4330

    accuracy                           0.95      8980
   macro avg       0.95      0.96      0.95      8980
weighted avg       0.95      0.95      0.95      8980

